<a href="https://colab.research.google.com/github/ImagingDataCommons/CloudSegmentator/blob/main/util/downloadBenchmarking/Notebooks/gcsVsS5cmdDownloadBenchmarkNotebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Benchmark: `gsutil`, `gcloud storage`, and `s5cmd` downloading from IDC's Google Cloud Storage bucket**

This notebook times how long it takes to download the same Imaging Data Commons (IDC) series from the **same `gs://` bucket** with three different tools:

- `gsutil -m cp`
- `gcloud storage cp`
- `s5cmd` (pointed at GCS's S3-compatible endpoint, since s5cmd only speaks the S3 protocol)

For every (tool, series) pair it records wall-clock download time, bytes downloaded, throughput, and CPU/RAM/disk usage, so the tools can be compared directly on identical data.

## Manifest input

You don't need a manifest to try this notebook. Leave `manifest_path` empty and it falls back to a built-in smoke-test sample of 3 small CT series (the same series already used as defaults in [downloadDicomAndConvertNotebook.ipynb](../../../workflows/TotalSegmentator/Notebooks/downloadDicomAndConvertNotebook.ipynb)).

When you do have a manifest, set `manifest_path` to one of:
- A local path (e.g. after uploading a file with the optional upload cell below)
- An `http(s)://` URL the notebook can fetch with `wget`

Manifest **contents** are auto-detected (`manifest_format="auto"`) among:
- A plain list of `gs://...` and/or `s3://...` URLs, one per line
- An s5cmd-style command manifest (lines like `cp s3://idc-open-data/<path> .`), as produced by the IDC portal's "s5cmd manifest" download option
- A CSV with a `SeriesInstanceUID` column (e.g. an IDC cohort manifest), or a CSV with an explicit URL column (`gcs_url`, `gs_url`, `aws_url`, `s3_url`, or `url`)

For manifest entries resolved through `idc-index`, this notebook treats each `SeriesInstanceUID` as a folder of objects and appends a `*` wildcard automatically. For entries parsed straight out of your manifest, the source path is used exactly as written (including any wildcard it already has) — so it works whether your manifest lists whole series or individual files.

If your manifest only has `s3://idc-open-data/...` URLs, the GCS counterpart is derived by swapping the scheme to `gs://idc-open-data/...` (IDC mirrors the same object keys under that bucket name on both clouds). A pre-flight check (below) verifies that assumption before the benchmark runs and tells you if it doesn't hold for your data — in that case set `gcs_bucket_override`.

## ⚠️ Costs and disk usage

Every series in the manifest is downloaded once **per tool**, so total network egress and disk usage are roughly `len(tools_to_benchmark)x` the manifest size. Start with a small `max_series` to sanity-check the setup before pointing this at a large manifest. `cleanup_after_each_download=True` (default) deletes each series right after it's timed so disk usage stays bounded regardless of manifest size.

In [ ]:
# Parameters
manifest_path = ""          # local path, http(s):// URL, or empty for the built-in smoke-test sample
manifest_format = "auto"    # "auto", "url_list", "s5cmd", or "csv"
max_series = 5              # cap how many series to benchmark; set to -1 to use the whole manifest
tools_to_benchmark = ["gcloud", "gsutil", "s5cmd"]
download_root = "benchmark_downloads"
cleanup_after_each_download = True   # delete each series right after timing it, to bound disk usage
gcs_bucket_override = None  # e.g. "idc-open-data" if the derived gs:// bucket name is wrong for your manifest
prompt_for_manifest_upload = False   # set True to get an upload widget when this cell runs in Colab


### Setup

In [ ]:
%%capture
import sys
if 'google.colab' in sys.modules:
    # gsutil and gcloud ship preinstalled in the Colab base image; only s5cmd needs installing.
    !pip install -q idc-index
    !wget -q "https://github.com/peak/s5cmd/releases/download/v2.2.2/s5cmd_2.2.2_Linux-64bit.tar.gz" \
        && tar -xzf s5cmd_2.2.2_Linux-64bit.tar.gz \
        && rm s5cmd_2.2.2_Linux-64bit.tar.gz \
        && mv s5cmd /usr/local/bin/s5cmd


In [ ]:
import io
import os
import re
import shutil
import subprocess
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from time import sleep
from urllib.parse import urlparse

import matplotlib.pyplot as plt
import pandas as pd
import psutil


In [ ]:
# Confirms gsutil, gcloud, and s5cmd are all reachable before running anything else.
tool_checks = {
    "gsutil": ["gsutil", "--version"],
    "gcloud": ["gcloud", "--version"],
    "s5cmd": ["s5cmd", "version"],
}
for name, cmd in tool_checks.items():
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=30)
        status = "OK" if result.returncode == 0 else f"exit {result.returncode}: {result.stderr.strip()[:200]}"
    except FileNotFoundError:
        status = "NOT FOUND"
    print(f"{name}: {status}")


If you have GCS HMAC keys (Cloud Storage console &rarr; Settings &rarr; Interoperability) for a GCP project, set them below so `s5cmd` authenticates instead of relying on anonymous public access. Leave commented out to try anonymous access first — the pre-flight check later in this notebook will tell you clearly if that fails.

In [ ]:
# os.environ["AWS_ACCESS_KEY_ID"] = "..."
# os.environ["AWS_SECRET_ACCESS_KEY"] = "..."


### (Optional) Upload a manifest

In [ ]:
if prompt_for_manifest_upload and 'google.colab' in sys.modules:
    from google.colab import files
    uploaded = files.upload()
    if uploaded:
        manifest_path = next(iter(uploaded))
        print(f"Using uploaded manifest: {manifest_path}")
else:
    print("Skipping upload (prompt_for_manifest_upload is False, or not running in Colab).")


In [ ]:
class MemoryMonitor:
    def __init__(self):
        # Flag to control the measurement loop
        self.keep_measuring = True
        # Get the path of the working disk
        self.working_disk_path = self.get_working_disk_path()

    def get_working_disk_path(self):
        # This code is specific to Terra/SB-CGC as multiple disks are mounted on the platforms

        # Get all disk partitions
        partitions = psutil.disk_partitions()
        for partition in partitions:
            # If root partition, return root path
            if partition.mountpoint == '/':
                return '/'
            # If cromwell_root is in mountpoint, return cromwell_root path
            elif '/cromwell_root' in partition.mountpoint:
                return '/cromwell_root'
        # Default to root directory if no specific path is found
        return '/'

    def measure_usage(self):
        # Initialize lists to store measurements
        cpu_usage = []
        ram_usage_mb = []
        disk_usage_all = []
        time_stamps = []

        # Record start time
        start_time = time.time()

        while self.keep_measuring:
            cpu_usage.append(psutil.cpu_percent())

            ram = psutil.virtual_memory()
            ram_usage_mb.append((ram.total - ram.available) / 1e6)

            disk_usage = psutil.disk_usage(self.working_disk_path)
            disk_usage_all.append(disk_usage.used / 1e9)

            time_stamps.append(time.time() - start_time)

            sleep(1)

        ram_total_mb = psutil.virtual_memory().total / 1e6
        disk_total = psutil.disk_usage(self.working_disk_path).total / 1e9
        return cpu_usage, ram_usage_mb, time_stamps, ram_total_mb, disk_usage_all, disk_total


### Parse the manifest into a table of GCS/S3 paths to benchmark

In [ ]:
def read_manifest_text(path: str) -> str:
    """Reads a manifest from a local path, downloading it first if it looks like a URL."""
    parsed = urlparse(path)
    if parsed.scheme in ("http", "https"):
        local_path = Path(Path(parsed.path).name or "manifest.txt")
        subprocess.run(["wget", "-q", "-O", str(local_path), path], check=True)
        path = str(local_path)
    return Path(path).read_text()


def detect_manifest_format(text: str) -> str:
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    if not lines:
        raise ValueError("Manifest is empty")
    first = lines[0]
    if first.startswith("cp "):
        return "s5cmd"
    if first.startswith(("gs://", "s3://")):
        return "url_list"
    return "csv"


def swap_scheme(url: str, target_scheme: str) -> str:
    """Converts a gs:// URL to s3:// (or vice versa) by swapping only the scheme.

    IDC mirrors the same object keys under the bucket name `idc-open-data` on both
    AWS and GCS, so a scheme swap is normally enough to go from one to the other.
    """
    rest = url.split("://", 1)[1]
    return f"{target_scheme}://{rest}"


def url_to_identifier(url: str) -> str:
    trimmed = url.rstrip("/*")
    return trimmed.rsplit("/", 1)[-1] or trimmed


def parse_s5cmd_manifest(text: str) -> list:
    """Extracts the source path from each `cp <source> <dest>` line, as written."""
    urls = []
    for line in text.splitlines():
        line = line.strip()
        if line.startswith("cp "):
            urls.append(line.split()[1])
    return urls


def parse_url_list(text: str) -> list:
    return [l.strip() for l in text.splitlines() if l.strip().startswith(("gs://", "s3://"))]


def build_series_table_from_urls(urls: list) -> pd.DataFrame:
    """Builds the benchmark table from URLs taken directly from the manifest (used as-is)."""
    rows = []
    for url in urls:
        scheme = url.split("://", 1)[0]
        aws_url = url if scheme == "s3" else swap_scheme(url, "s3")
        gcs_url = url if scheme == "gs" else swap_scheme(url, "gs")
        rows.append({"identifier": url_to_identifier(url), "aws_url": aws_url, "gcs_url": gcs_url})
    return pd.DataFrame(rows)


In [ ]:
def resolve_series_uids_with_idc_index(series_uids: list, idc_client) -> pd.DataFrame:
    """Looks up SeriesInstanceUIDs in idc-index and derives gs:// / s3:// series paths.

    idc-index's schema has changed across releases, so the relevant URL column(s) are
    discovered at runtime instead of hardcoding a column name. See
    https://github.com/ImagingDataCommons/idc-index/blob/main/queries/idc_index.sql
    Each resolved series is treated as a folder of objects, so a `*` wildcard is appended.
    """
    sample_cols = idc_client.sql_query("SELECT * FROM index LIMIT 1").columns
    aws_col = next((c for c in sample_cols if "aws" in c.lower() and "url" in c.lower()), None)
    gcs_col = next((c for c in sample_cols if ("gcs" in c.lower() or "gcp" in c.lower()) and "url" in c.lower()), None)
    if not aws_col and not gcs_col:
        raise RuntimeError(
            "Could not find an AWS/GCS URL column in the idc-index `index` table. "
            f"Available columns: {list(sample_cols)}. Supply gs:// URLs directly via "
            "manifest_format='url_list' instead."
        )

    select_cols = ["SeriesInstanceUID"] + [c for c in (aws_col, gcs_col) if c]
    uid_list_sql = ", ".join(f"'{uid}'" for uid in series_uids)
    df = idc_client.sql_query(f"""
        SELECT {", ".join(select_cols)}
        FROM index
        WHERE SeriesInstanceUID IN ({uid_list_sql})
    """)

    rename_map = {"SeriesInstanceUID": "identifier"}
    if aws_col:
        rename_map[aws_col] = "aws_url"
    if gcs_col:
        rename_map[gcs_col] = "gcs_url"
    df = df.rename(columns=rename_map)

    if "gcs_url" not in df.columns:
        df["gcs_url"] = df["aws_url"].apply(lambda u: swap_scheme(u, "gs"))
    if "aws_url" not in df.columns:
        df["aws_url"] = df["gcs_url"].apply(lambda u: swap_scheme(u, "s3"))

    df["aws_url"] = df["aws_url"].astype(str).str.rstrip("/") + "/*"
    df["gcs_url"] = df["gcs_url"].astype(str).str.rstrip("/") + "/*"
    return df[["identifier", "aws_url", "gcs_url"]]


def get_default_smoke_test_series() -> list:
    # Confirmed-working public IDC CT SeriesInstanceUIDs, reused from this repo's
    # downloadDicomAndConvertNotebook.ipynb default sample.
    return [
        '1.3.6.1.4.1.14519.5.2.1.7009.9004.100143549999116733615345241533',
        '1.3.6.1.4.1.14519.5.2.1.7009.9004.100148350742920339334061834697',
        '1.3.6.1.4.1.14519.5.2.1.7009.9004.100241427395754063917290539621',
    ]


def build_benchmark_table(manifest_path, manifest_format, idc_client, max_series, gcs_bucket_override=None) -> pd.DataFrame:
    if not manifest_path:
        print("No manifest_path given - using the built-in smoke-test sample of 3 series.")
        series_table = resolve_series_uids_with_idc_index(get_default_smoke_test_series(), idc_client)
    else:
        text = read_manifest_text(manifest_path)
        fmt = manifest_format if manifest_format != "auto" else detect_manifest_format(text)
        if fmt == "s5cmd":
            series_table = build_series_table_from_urls(parse_s5cmd_manifest(text))
        elif fmt == "url_list":
            series_table = build_series_table_from_urls(parse_url_list(text))
        elif fmt == "csv":
            csv_df = pd.read_csv(io.StringIO(text))
            url_col = next((c for c in csv_df.columns if c.lower() in ("gcs_url", "gs_url", "aws_url", "s3_url", "url")), None)
            if url_col:
                series_table = build_series_table_from_urls(csv_df[url_col].dropna().tolist())
            elif "SeriesInstanceUID" in csv_df.columns:
                series_table = resolve_series_uids_with_idc_index(csv_df["SeriesInstanceUID"].dropna().tolist(), idc_client)
            else:
                raise ValueError(
                    "Could not find a URL or SeriesInstanceUID column in the CSV manifest. "
                    f"Columns found: {list(csv_df.columns)}"
                )
        else:
            raise ValueError(f"Unrecognized manifest format: {fmt}")

    if gcs_bucket_override:
        series_table["gcs_url"] = series_table["gcs_url"].apply(
            lambda u: re.sub(r"^gs://[^/]+", f"gs://{gcs_bucket_override}", u)
        )

    if max_series is not None and max_series > 0:
        series_table = series_table.head(max_series)

    return series_table.reset_index(drop=True)


In [ ]:
from idc_index import index

idc_client = index.IDCClient()

effective_max_series = None if max_series in (None, -1) else max_series
series_table = build_benchmark_table(manifest_path, manifest_format, idc_client, effective_max_series, gcs_bucket_override)
assert len(series_table) > 0, "No series resolved - check manifest_path/manifest_format."
print(f"Resolved {len(series_table)} series to benchmark.")
series_table


### Pre-flight checks

Verifies the derived `gs://` paths actually resolve, and that `s5cmd` can reach the GCS endpoint, before spending time on the full benchmark.

In [ ]:
def verify_gcs_path(gcs_url: str) -> bool:
    result = subprocess.run(["gsutil", "ls", gcs_url], capture_output=True, text=True, timeout=60)
    return result.returncode == 0 and bool(result.stdout.strip())


def s5cmd_base_command() -> list:
    cmd = ["s5cmd"]
    if not os.environ.get("AWS_ACCESS_KEY_ID"):
        cmd.append("--no-sign-request")
    cmd += ["--endpoint-url", "https://storage.googleapis.com"]
    return cmd


bad_gcs_paths = [u for u in series_table["gcs_url"] if not verify_gcs_path(u)]
if bad_gcs_paths:
    print("WARNING: gsutil could not list the following derived gs:// paths. The "
          "scheme-swap assumption (s3://idc-open-data/... -> gs://idc-open-data/...) "
          "may not hold for this manifest. Set gcs_bucket_override or supply gs:// URLs directly.")
    for u in bad_gcs_paths:
        print(" -", u)
else:
    print("All derived gs:// paths resolved successfully.")

probe_cmd = s5cmd_base_command() + ["ls", series_table.iloc[0]["aws_url"]]
probe = subprocess.run(probe_cmd, capture_output=True, text=True, timeout=60)
if probe.returncode == 0:
    print("s5cmd can reach the GCS endpoint.")
else:
    print("WARNING: s5cmd could not list objects via the GCS S3-compatible endpoint:")
    print(" ", probe.stderr.strip())
    print("If this is an auth error, generate HMAC keys for a GCP project under Cloud "
          "Storage > Settings > Interoperability, then set AWS_ACCESS_KEY_ID / "
          "AWS_SECRET_ACCESS_KEY in the Setup section above and re-run from there.")


### Download + timing functions for each tool

In [ ]:
def source_url_for_tool(row: pd.Series, tool: str) -> str:
    # s5cmd only speaks the S3 protocol, so it needs the s3:// form even though
    # --endpoint-url points it at GCS. gsutil/gcloud need the gs:// form.
    return row["aws_url"] if tool == "s5cmd" else row["gcs_url"]


def build_download_command(tool: str, source_url: str, dest_dir: str) -> list:
    if tool == "gsutil":
        return ["gsutil", "-m", "cp", "-r", source_url, f"{dest_dir}/"]
    if tool == "gcloud":
        return ["gcloud", "storage", "cp", "-r", source_url, f"{dest_dir}/"]
    if tool == "s5cmd":
        return s5cmd_base_command() + ["cp", source_url, f"{dest_dir}/"]
    raise ValueError(f"Unknown tool: {tool}")


def dir_size_bytes(path: Path) -> int:
    return sum(f.stat().st_size for f in path.rglob("*") if f.is_file())


def time_download(tool: str, row: pd.Series, download_root: str, cleanup: bool) -> dict:
    dest_dir = Path(download_root) / tool / row["identifier"]
    if dest_dir.exists():
        shutil.rmtree(dest_dir)
    dest_dir.mkdir(parents=True, exist_ok=True)

    source_url = source_url_for_tool(row, tool)
    cmd = build_download_command(tool, source_url, str(dest_dir))

    start_time = time.time()
    result = subprocess.run(cmd, capture_output=True, text=True)
    download_time = time.time() - start_time

    bytes_downloaded = dir_size_bytes(dest_dir) if result.returncode == 0 else 0
    record = {
        "tool": tool,
        "identifier": row["identifier"],
        "download_time_sec": download_time,
        "bytes_downloaded": bytes_downloaded,
        "throughput_MBps": (bytes_downloaded / 1e6 / download_time) if download_time > 0 else 0,
        "exit_code": result.returncode,
        "stderr_tail": result.stderr.strip()[-500:] if result.returncode != 0 else "",
    }

    if cleanup:
        shutil.rmtree(dest_dir, ignore_errors=True)

    return record


### Run the benchmark

In [ ]:
runtime_stats = pd.DataFrame()

for _, row in series_table.iterrows():
    for tool in tools_to_benchmark:
        print(f"\n[{tool}] downloading {row['identifier']} ...")
        with ThreadPoolExecutor() as executor:
            monitor = MemoryMonitor()
            mem_future = executor.submit(monitor.measure_usage)
            try:
                dl_future = executor.submit(time_download, tool, row, download_root, cleanup_after_each_download)
                record = dl_future.result()
            finally:
                monitor.keep_measuring = False
                cpu_usage, ram_usage_mb, time_stamps, ram_total_mb, disk_usage_all, disk_total = mem_future.result()

        record["cpu_usage_avg"] = sum(cpu_usage) / len(cpu_usage) if cpu_usage else 0
        record["ram_usage_mb_avg"] = sum(ram_usage_mb) / len(ram_usage_mb) if ram_usage_mb else 0

        status = "OK" if record["exit_code"] == 0 else f"FAILED ({record['stderr_tail']})"
        print(f"  {status} - {record['download_time_sec']:.1f}s, "
              f"{record['bytes_downloaded'] / 1e6:.1f} MB, "
              f"{record['throughput_MBps']:.2f} MB/s")

        runtime_stats = pd.concat([runtime_stats, pd.DataFrame([record])], ignore_index=True)

runtime_stats


### Results

In [ ]:
ok = runtime_stats[runtime_stats.exit_code == 0]
failed = runtime_stats[runtime_stats.exit_code != 0]

summary = ok.groupby("tool")[["download_time_sec", "throughput_MBps"]].agg(["mean", "median", "min", "max"])
summary


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ok.boxplot(column="download_time_sec", by="tool", ax=ax1)
ax1.set_title("Download time per series")
ax1.set_xlabel("")
ax1.set_ylabel("seconds")

ok.boxplot(column="throughput_MBps", by="tool", ax=ax2)
ax2.set_title("Throughput per series")
ax2.set_xlabel("")
ax2.set_ylabel("MB/s")

plt.suptitle("")
plt.tight_layout()
plt.show()


In [ ]:
if not failed.empty:
    print("Some downloads failed and were excluded from the summary/plots above:")
    display(failed[["tool", "identifier", "exit_code", "stderr_tail"]])


In [ ]:
runtime_stats.to_csv("gcs_download_benchmark_results.csv", index=False)
print("Saved gcs_download_benchmark_results.csv")


### Cleanup

In [ ]:
if Path(download_root).exists():
    shutil.rmtree(download_root)
    print(f"Removed {download_root}/")
